In [ ]:
modello_per_analisi = 'Opel_Corsa'

### 0. Import funzioni e librerie

In [ ]:
import os
import sys
from pathlib import Path
import importlib
import requests
import pandas as pd
import joblib

import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
import dash_bootstrap_components as dbc

In [ ]:
from vehicle_price_monitor.paths import ROOT, RAW, PROCESSED, MODELS, GEO, load_config
project_root = ROOT
from app.dashboard import get_data_dummy_for_dash, create_dashboard

In [ ]:
config = load_config(modello_per_analisi)
# --- Accesso ai parametri di configurazione ---
#allestimenti = config.allestimenti
allestimento_performance = config.allestimento_performance
allestimento_sport = config.allestimento_sport
allestimento_middle = config.allestimento_middle
allestimento_base = config.allestimento_base
motorizzazioni_norm = config.motorizzazioni_norm
modelli_ord = config.modelli_ord
mappa_cv = config.mappa_cv
mappa_allestimenti = config.mappa_allestimenti

### 1. Dati

In [ ]:
project_root = Path.cwd().parents[0]

model_ml = joblib.load(MODELS / modello_per_analisi / 'modello_rf.pkl')
scaler_ml = joblib.load(MODELS / modello_per_analisi / 'scaler_rf.pkl')
mse = joblib.load(MODELS / modello_per_analisi / 'mse_rf.pkl')
rmse = joblib.load(MODELS / modello_per_analisi / 'rmse_rf.pkl')
r2 = joblib.load(MODELS / modello_per_analisi / 'r2_rf.pkl')
cols_model = joblib.load(MODELS / modello_per_analisi / 'cols_model.pkl')

#### CARICAMENTO DATI SINGOLI
data_dummy = pd.read_csv(PROCESSED / modello_per_analisi / f'data_dummy_{modello_per_analisi}.csv')
data_originale = pd.read_csv(PROCESSED / modello_per_analisi / f'data_{modello_per_analisi}.csv')

In [ ]:
allestimenti = list(set(allestimento_performance + allestimento_sport + allestimento_middle + allestimento_base))
carburanti = data_originale['Carburante'].unique().tolist()
val_metrics = dict(mse=mse,rmse=rmse,r2=r2)

cols_to_remove = ['prezzo_previsto', 'delta_prezzo', 'is_conveniente']
cols_model = [col for col in cols_model if col not in cols_to_remove]

### 2. Dashboard di previsione

In [ ]:
# ===============================
# Esempio di utilizzo
# ===============================
app = create_dashboard(modello_per_analisi, data_originale, allestimenti, carburanti, allestimento_performance,
                       allestimento_sport, allestimento_middle, model_ml, scaler_ml, val_metrics, cols_model, get_data_dummy_for_dash)

app.run(debug=True, port=8051)